In [ ]:
import plotly.graph_objects as go
import numpy as np

# Define the function
def f(x, y):
    return (x**11 + y**11)**(1/11)

# Create a grid of x and y values
n = 200
x = np.linspace(-2, 2, n)
y = np.linspace(-2, 2, n)
X, Y = np.meshgrid(x, y)

# Calculate the z values
Z = np.zeros_like(X)
for i in range(n):
    for j in range(n):
        if X[i, j] == 0 and Y[i, j] == 0:
            Z[i, j] = 0  # Define the value at (0, 0)
        else:
            Z[i, j] = f(X[i, j], Y[i, j])

# Create the surface plot
fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y)])

# Update the layout for better visualization
fig.update_layout(
    #title='Interactive Plot of $f(x, y) = (x^11 + y^11)^(1/11)$',
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        aspectratio=dict(x=1.25, y=1.25, z=1.5),
        camera=dict(eye=dict(x=-1.5, y=-1.5, z=1.5)),
        # Move 3D annotations into the scene dictionary
        annotations=[
            dict(
                x=0,
                y=0,
                z=0.2,
                text='Sharp point at (0, 0) indicates non-differentiability',
                showarrow=True,
                arrowhead=1,
                ax=50,
                ay=-50
            ),
            #dict(
            #    x=0.8,
            #    y=0.8,
            #    z=f(0.8, 0.8),
            #    text='Function f(x, y) = (x³ + y³)^(1/3)',
            #    showarrow=False,
            #    xanchor='left',
            #    yanchor='top'
            #)
        ]
    ),
    height=600
)


fig.update_layout(
    margin=dict(l=20, r=20, t=20, b=20),
)
fig.show()

In [ ]:
fig.write_html("non-diff-but-der.html")

In [ ]:
# Add text explanations (2D annotations on the paper/margin)
#fig.add_annotation(
#    text="<b>Partial Derivatives at (0, 0):</b><br>"
#         "∂f/∂x |<sub>(0,0)</sub> = lim (h→0) [f(h, 0) - f(0, 0)] / h = 1<br>"
#         "∂f/∂y |<sub>(0,0)</sub> = lim (k→0) [f(0, k) - f(0, 0)] / k = 1",
#    xref="paper", yref="paper",
#    x=0.05, y=0.95,
#    showarrow=False
#)

#fig.add_annotation(
#    text="<b>Non-Differentiability at (0, 0):</b><br>"
#         "The sharp cusp means the tangent plane is undefined here.<br>"
#         "Directional derivatives vary by direction, so the function isn't differentiable.",
#    xref="paper", yref="paper",
#    x=0.05, y=0.80,
#    showarrow=False
#)

In [ ]:
import plotly.io as pio


# Convert the figure to HTML
html_string = pio.to_html(fig, full_html=False, include_plotlyjs="cdn")

# Create a simple HTML page with the embedded plot
embeddable_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Embedded Plotly Plot</title>
</head>
<body>
    {html_string}
</body>
</html>
"""

# Save the HTML to a file (optional)
with open("non-diff-but-der2.html", "w") as f:
    f.write(embeddable_html)

print(embeddable_html) # print the html to the console, to be used in an iframe.


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
from ipywidgets import interact

def cobb_douglas(x, y, alpha=0.5):
    return (x ** alpha) * (y ** (1 - alpha))

def plot_indifference_curve(utility_level=1.0):
    x = np.linspace(0.1, 10, 100)
    y = (utility_level / (x ** 0.5)) ** (1 / 0.5)  # Solving for y in U(x,y) = level
    
    X, Y = np.meshgrid(np.linspace(0.1, 10, 50), np.linspace(0.1, 10, 50))
    Z = cobb_douglas(X, Y)
    
    fig = sp.make_subplots(rows=1, cols=2, subplot_titles=("3D Utility Function", "2D Indifference Curve"),
                           specs=[[{'type': 'surface'}, {'type': 'xy'}]])
    
    # 3D Surface Plot
    fig.add_trace(go.Surface(z=Z, x=X, y=Y, colorscale='Viridis', opacity=0.7), row=1, col=1)
    
    # Add plane at utility_level
    plane_z = np.full_like(Z, utility_level)
    fig.add_trace(go.Surface(z=plane_z, x=X, y=Y, colorscale='Reds', opacity=0.5, showscale=False), row=1, col=1)
    fig.update_scenes(camera=dict(eye=dict(x=-2, y=-2, z=1)))
    
    # Compute color for the indifference curve based on Viridis colormap
    from matplotlib import colormaps
    viridis = colormaps.get_cmap('viridis')
    norm_utility = (utility_level - np.min(Z)) / (np.max(Z) - np.min(Z))  # Normalize utility level
    curve_color = f'rgba({int(viridis(norm_utility)[0] * 255)}, {int(viridis(norm_utility)[1] * 255)}, {int(viridis(norm_utility)[2] * 255)}, 1)'
    
    # 2D Indifference Curve with color matching the 3D scale
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color=curve_color), name=f'Utility = {utility_level}'), row=1, col=2)
    
    # Fix axis scale for 2D plot
    fig.update_xaxes(range=[0, 10], row=1, col=2)
    fig.update_yaxes(range=[0, 10], row=1, col=2)
    
    fig.update_layout(title_text=f"Indifference Curve as a Level Curve of Utility (U = {utility_level})",
                      height=600, width=1000)
    

    fig.show()

# Interactive slider
interact(plot_indifference_curve, utility_level=(0.5, 5.0, 0.1))


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.io as pio
import dash
from dash import Dash, dcc, html


def cobb_douglas(x, y, alpha=0.5):
    return (x ** alpha) * (y ** (1 - alpha))

def generate_figure(utility_level=1.0):
    x = np.linspace(0.1, 10, 100)
    y = (utility_level / (x ** 0.5)) ** (1 / 0.5)
    
    X, Y = np.meshgrid(np.linspace(0.1, 10, 50), np.linspace(0.1, 10, 50))
    Z = cobb_douglas(X, Y)
    
    fig = sp.make_subplots(rows=1, cols=2, subplot_titles=("3D Utility Function", "2D Indifference Curve"),
                           specs=[[{'type': 'surface'}, {'type': 'xy'}]])
    
    surface = go.Surface(z=Z, x=X, y=Y, colorscale='Viridis', opacity=0.7)
    fig.add_trace(surface, row=1, col=1)
    
    plane_z = np.full_like(Z, utility_level)
    fig.add_trace(go.Surface(z=plane_z, x=X, y=Y, colorscale='Reds', opacity=0.5, showscale=False), row=1, col=1)
    
    fig.update_scenes(camera=dict(eye=dict(x=2, y=2, z=1)))
    
    from matplotlib import colormaps
    viridis = colormaps.get_cmap('viridis')
    norm_utility = (utility_level - np.min(Z)) / (np.max(Z) - np.min(Z))
    curve_color = f'rgba({int(viridis(norm_utility)[0] * 255)}, {int(viridis(norm_utility)[1] * 255)}, {int(viridis(norm_utility)[2] * 255)}, 1)'
    
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color=curve_color), name=f'Utility = {utility_level}'), row=1, col=2)
    
    fig.update_xaxes(range=[0, 10], row=1, col=2)
    fig.update_yaxes(range=[0, 10], row=1, col=2)
    
    fig.update_layout(title_text=f"Indifference Curve as a Level Curve of Utility (U = {utility_level})",
                      height=600, width=1000)
    
    return fig

app = Dash(__name__)

app.layout = html.Div([
    dcc.Slider(
        id='utility-slider',
        min=0.5,
        max=5.0,
        step=0.1,
        value=1.0,
        marks={i: str(i) for i in range(1, 6)}
    ),
    dcc.Graph(id='utility-graph', figure=generate_figure(1.0))
])

@app.callback(
    dash.dependencies.Output('utility-graph', 'figure'),
    [dash.dependencies.Input('utility-slider', 'value')]
)
def update_figure(utility_level):
    return generate_figure(utility_level)

if __name__ == '__main__':
    app.run(debug=False)
    
# Convert app layout to HTML for static embedding
html_string = app.index_string
with open("indifference_curve_plot.html", "w") as f:
    f.write(html_string)

print("HTML file saved: indifference_curve_plot.html")


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Define the original function
def original_function(x, y):
    return np.sin(np.sqrt(x**2 + y**2)) / (np.sqrt(x**2 + y**2) + 1)

# 2. Define the second-order Taylor approximation function
def taylor_approximation(x, y, a, b):
    # Calculate the function value and derivatives at (a, b)
    f_ab = original_function(a, b)

    # First-order partial derivatives
    h = 1e-6  # Small step for numerical differentiation
    df_dx = (original_function(a + h, b) - original_function(a - h, b)) / (2 * h)
    df_dy = (original_function(a, b + h) - original_function(a, b - h)) / (2 * h)

    # Second-order partial derivatives
    df_dxx = (original_function(a + h, b) - 2 * original_function(a, b) + original_function(a - h, b)) / (h**2)
    df_dyy = (original_function(a, b + h) - 2 * original_function(a, b) + original_function(a, b - h)) / (h**2)
    df_dxy = (original_function(a + h, b + h) - original_function(a + h, b - h) - original_function(a - h, b + h) + original_function(a - h, b - h)) / (4 * h**2)

    # Second-order Taylor approximation formula
    return (f_ab +
            df_dx * (x - a) +
            df_dy * (y - b) +
            0.5 * df_dxx * (x - a)**2 +
            df_dxy * (x - a) * (y - b) +
            0.5 * df_dyy * (y - b)**2)

# 3. Generate data points for plotting
x_range = np.linspace(-5, 5, 50)
y_range = np.linspace(-5, 5, 50)
x_grid, y_grid = np.meshgrid(x_range, y_range)
z_original = original_function(x_grid, y_grid)

# 4. Initial point for Taylor approximation
a_initial = 1
b_initial = 1
z_taylor_initial = taylor_approximation(x_grid, y_grid, a_initial, b_initial)

# 5. Create the figure with subplots
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'surface'}, {'type': 'surface'}]],
                    subplot_titles=('Original Function', 'Taylor Approximation'))

# Add the original function plot
original_trace = go.Surface(z=z_original, x=x_range, y=y_range, name='Original')
fig.add_trace(original_trace, row=1, col=1)

# Add the Taylor approximation plot (initially)
taylor_trace = go.Surface(z=z_taylor_initial, x=x_range, y=y_range, name='Taylor Approx.')
fig.add_trace(taylor_trace, row=1, col=2)

# 6. Add sliders for interactive control of the approximation point (a, b)
a_slider = {
    'pad': {'b': 10, 't': 60},
    'len': 0.9,
    'x': 0.1,
    'y': 0,
    'currentvalue': {'prefix': 'a = '},
    'steps': []
}
b_slider = {
    'pad': {'b': 10, 't': 60},
    'len': 0.9,
    'x': 0.1,
    'y': 0.1,
    'currentvalue': {'prefix': 'b = '},
    'steps': []
}

# Define the range of values for a and b in the sliders
a_values = np.linspace(-3, 3, 20)
b_values = np.linspace(-3, 3, 20)

for a in a_values:
    step = {
        'method': 'update',
        'label': f'{a:.2f}',
        'value': str(a),
        'args': [{'surface': [None, {'z': taylor_approximation(x_grid, y_grid, a, b_initial)}]}, [1]],  # Update the second trace
    }
    a_slider['steps'].append(step)

for b in b_values:
    step = {
        'method': 'update',
        'label': f'{b:.2f}',
        'value': str(b),
        'args': [{'surface': [None, {'z': taylor_approximation(x_grid, y_grid, a_initial, b)}]}, [1]],  # Update the second trace
    }
    b_slider['steps'].append(step)

# Initially set the 'args' of the first slider to use the initial b value
for step in a_slider['steps']:
    a_val = float(step['value'])
    step['args'] = [{'surface': [None, {'z': taylor_approximation(x_grid, y_grid, a_val, b_initial)}]}, [1]]

# Initially set the 'args' of the second slider to use the initial a value
for step in b_slider['steps']:
    b_val = float(step['value'])
    step['args'] = [{'surface': [None, {'z': taylor_approximation(x_grid, y_grid, a_initial, b_val)}]}, [1]]

fig.update_layout(
    title_text='Original Function and its Second-Order Taylor Approximation',
    scene1=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='f(x, y)'),
    scene2=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Taylor Approx.'),
    sliders=[a_slider, b_slider]
)

fig.show()

In [5]:
import numpy as np
import plotly.graph_objects as go

# 1. Define a cubic original function
def original_function(x, y):
    return x**3 + y**3

# 2. Define the second-order Taylor approximation function
def taylor_approximation(x, y, a, b):
    # Function value at (a, b)
    f_ab = a**3 + b**3

    # First-order partial derivatives
    df_dx = 3 * a**2
    df_dy = 3 * b**2

    # Second-order partial derivatives
    df_dxx = 6 * a
    df_dyy = 6 * b
    df_dxy = 0

    # Second-order Taylor approximation formula
    return (f_ab +
            df_dx * (x - a) +
            df_dy * (y - b) +
            0.5 * df_dxx * (x - a)**2 +
            df_dxy * (x - a) * (y - b) +
            0.5 * df_dyy * (y - b)**2)

# 3. Choose the point for Taylor approximation
a_approx = 1.0
b_approx = 1.0
z_approx = original_function(a_approx, b_approx)

# 4. Generate data points for plotting
x_range = np.linspace(-3, 3, 50)  # Widened x-range
y_range = np.linspace(-3, 3, 50)  # Widened y-range
x_grid, y_grid = np.meshgrid(x_range, y_range)
z_original = original_function(x_grid, y_grid)
z_taylor = taylor_approximation(x_grid, y_grid, a_approx, b_approx)

# 5. Create the 3D plot
fig = go.Figure(data=[
    go.Surface(z=z_original, x=x_grid, y=y_grid, name='Original Function', colorscale='Viridis'),
    go.Surface(z=z_taylor, x=x_grid, y=y_grid, name=f'Taylor Approx. (a={a_approx:.2f}, b={b_approx:.2f})',
               opacity=0.7,
               surfacecolor=np.full(z_taylor.shape, 'red')),  # Single color for approximation
    go.Scatter3d(x=[a_approx], y=[b_approx], z=[z_approx],
                 mode='markers',
                 marker=dict(size=8, color='red'),
                 name='Approximation Point')
])

# 6. Update the layout for better visualization
fig.update_layout(
    title=f'Original Function (x³ + y³) and its Second-Order Taylor Approximation around (a={a_approx:.2f}, b={b_approx:.2f})',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        xaxis=dict(range=[-3, 3]),  # Explicitly set x-axis range
        yaxis=dict(range=[-3, 3])   # Explicitly set y-axis range
    ),
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()

In [19]:
import numpy as np
import plotly.graph_objects as go

# 1. Define the ln original function
def original_function(x, y):
    return np.log(x**2 + y**2 + 1)

# 2. Define the second-order Taylor approximation function
def taylor_approximation(x, y, a, b):
    # Function value at (a, b)
    f_ab = np.log(a**2 + b**2 + 1)

    # First-order partial derivatives
    df_dx = (2 * a) / (a**2 + b**2 + 1)
    df_dy = (2 * b) / (a**2 + b**2 + 1)

    # Second-order partial derivatives
    denominator = (a**2 + b**2 + 1)
    df_dxx = (2 * denominator - (2 * a) * (2 * a)) / (denominator**2)
    df_dyy = (2 * denominator - (2 * b) * (2 * b)) / (denominator**2)
    df_dxy = -(2 * a) * (2 * b) * 2 / (denominator**2)

    # Second-order Taylor approximation formula
    return (f_ab +
            df_dx * (x - a) +
            df_dy * (y - b) +
            0.5 * df_dxx * (x - a)**2 +
            df_dxy * (x - a) * (y - b) +
            0.5 * df_dyy * (y - b)**2)

# 3. Choose the point for Taylor approximation
a_approx = 0.0
b_approx = 0.0
z_approx = original_function(a_approx, b_approx)

# 4. Generate data points for plotting
x_range = np.linspace(-3, 3, 50)   # Widened x-range
y_range = np.linspace(-3, 3, 50)   # Widened y-range
x_grid, y_grid = np.meshgrid(x_range, y_range)
z_original = original_function(x_grid, y_grid)
z_taylor = taylor_approximation(x_grid, y_grid, a_approx, b_approx)

# 5. Create the 3D plot
fig = go.Figure(data=[
    go.Surface(z=z_original, x=x_grid, y=y_grid, name='Original Function (ln(x² + y² + 1))', colorscale='Viridis'),
    go.Surface(z=z_taylor, x=x_grid, y=y_grid, name=f'Taylor Approx. (a={a_approx:.2f}, b={b_approx:.2f})',
               opacity=0.7,
               colorscale=[[0, '#FFCCCC'], [1, '#FFCCCC']],
               showscale=False),  # Single color for approximation using colorscale with hex code
    go.Scatter3d(x=[a_approx], y=[b_approx], z=[z_approx],
                 mode='markers',
                 marker=dict(size=8, color='red'),
                 name='Approximation Point')
])

# 6. Update the layout for better visualization
fig.update_layout(
    title=f'Original Function (ln(x² + y² + 1)) and its Second-Order Taylor Approximation around (a={a_approx:.2f}, b={b_approx:.2f})',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        xaxis=dict(range=[-3, 3]),   # Explicitly set x-axis range
        yaxis=dict(range=[-3, 3]),    # Explicitly set y-axis range
        zaxis=dict(range=[None, 6.5])   # Set max z to 10
    ),
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Define the ln original function
def original_function(x, y):
    return np.log(x**2 + y**2 + 1)

# 2. Define the second-order Taylor approximation function
def taylor_approximation(x, y, a, b):
    # Function value at (a, b)
    f_ab = np.log(a**2 + b**2 + 1)

    # First-order partial derivatives
    df_dx = (2 * a) / (a**2 + b**2 + 1)
    df_dy = (2 * b) / (a**2 + b**2 + 1)

    # Second-order partial derivatives
    denominator = (a**2 + b**2 + 1)
    df_dxx = (2 * denominator - (2 * a) * (2 * a)) / (denominator**2)
    df_dyy = (2 * denominator - (2 * b) * (2 * b)) / (denominator**2)
    df_dxy = -(2 * a) * (2 * b) * 2 / (denominator**2)

    # Second-order Taylor approximation formula
    return (f_ab +
            df_dx * (x - a) +
            df_dy * (y - b) +
            0.5 * df_dxx * (x - a)**2 +
            df_dxy * (x - a) * (y - b) +
            0.5 * df_dyy * (y - b)**2)

# 3. Choose the point for Taylor approximation
a_approx = 0.0
b_approx = 0.0
z_approx = original_function(a_approx, b_approx)

# 4. Generate data points for plotting
x_range = np.linspace(-3, 3, 50)  # Widened x-range
y_range = np.linspace(-3, 3, 50)  # Widened y-range
x_grid, y_grid = np.meshgrid(x_range, y_range)
z_original = original_function(x_grid, y_grid)
z_taylor = taylor_approximation(x_grid, y_grid, a_approx, b_approx)

# 5. Create the 3D plot
fig = go.Figure(data=[
    go.Surface(z=z_original, x=x_grid, y=y_grid, name='Original Function (ln(x² + y² + 1))', colorscale='Viridis'),
    go.Surface(z=z_taylor, x=x_grid, y=y_grid, name=f'Taylor Approx. (a={a_approx:.2f}, b={b_approx:.2f})',
               opacity=0.7,
               surfacecolor=np.full(z_taylor.shape, 'red')),  # Single color for approximation
    go.Scatter3d(x=[a_approx], y=[b_approx], z=[z_approx],
                 mode='markers',
                 marker=dict(size=8, color='red'),
                 name='Approximation Point')
])

# 6. Update the layout for better visualization
fig.update_layout(
    title=f'Original Function (ln(x² + y² + 1)) and its Second-Order Taylor Approximation around (a={a_approx:.2f}, b={b_approx:.2f})',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        xaxis=dict(range=[-3, 3]),  # Explicitly set x-axis range
        yaxis=dict(range=[-3, 3]),   # Explicitly set y-axis range
        zaxis=dict(type="log")  # Set z-axis to log scale
    ),
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()